# QuakeMind — Drone sahnelerinde en yüksek çözünürlükte hasar maskesi (Colab T4)

**Ne yapar:** Modelin uygulamada test edildiği üç **Help.NGO drone** sahnesinde
(deprem sonrası, 9 Şubat 2023) SegFormer hasar maskesi üretir:

- **H-D1** Defne - Elektrik Mah. — Help.NGO, 2023-02-09, 3.4 cm/px
- **H-D2** Defne - Turunclu / Akdeniz / Armutlu Mah. — Help.NGO, Portugese GRU, 2023-02-09, 3.9 cm/px
- **K-D1** Merkez - 12 Subat Stadyumu cevresi — Help.NGO, 2023-02-09, 3.8 cm/px

Yerelde (CPU) **z=18, 19, 20** zaten üretildi. Bu defter **z=21 (~6 cm/px)** ve
**z=22 (~3 cm/px, sahnelerin doğal çözünürlüğü)** için. z=22'nin üstü yeni bilgi
katmaz; karo sunucusu yalnızca büyütür.

**Dikkat — en yüksek zoom en iyi sonuç demek değil.** Model KATE-CD ile
(Maxar/Pléiades, **0,3–0,5 m/px**) eğitildi. z=22'de bir bina, modelin eğitimde
gördüğünden ~15 kat büyük görünür. Hangi zoom'un en iyi olduğuna yerelde
`benchmark/validate_model_masks.py` ile **EMSR648'e karşı ölçerek** karar
verilecek.

**Nasıl çalıştırılır**
1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**
2. **Çalışma zamanı → Tümünü çalıştır**
3. İki kez `model_masks.zip` iner (z=21 bitince ve z=22 bitince). **İkincisini** sakla;
   z=22 bellek yüzünden çökerse **birincisi** elinde kalır.
4. Zip'i yerelde **`QuakeMindBackend/benchmark/data/model_masks/`** içine aç —
   `model_masks/H-D1-z22/damage_mask.png` gibi klasörler oluşmalı.
5. Yerelde:
   `python benchmark/validate_model_masks.py --dataset drone --suffixes=-z18,-z19,-z20,-z21,-z22`

**Tahmini süre (T4):** z=21 ~10-15 dk, z=22 ~30-60 dk.
**Bellek:** z=22'de tek sahne ~370 megapiksel; çıkarım ~9 GB RAM ister. Ücretsiz
Colab (~12,7 GB) sınırda. Çökerse z=21 sonuçlarıyla devam edilebilir.

**Değiştirme:** Bu defter `benchmark/colab/build_colab_notebook.py` ile üretildi.
Kodu burada düzenleme; depoda düzenleyip defteri yeniden üret.

Gömülü dosyaların SHA-256 parmak izleri:
```
{
  "inference.py": "9204c92a668f52daec9f6178efe412c0661b80eb51b8a3fa4604179c8a6c2724",
  "make_model_masks.py": "3ebf39cf4536d81dd3b910fce612e7e17bd2ee120a1bda3f4527df40f5fd7b1e",
  "qm_geo.py": "199cf916abb00aa17d3238e5feca29f7d367f7ec957dce3546960d0f90fbba6a",
  "zones_drone.json": "6aa623217d28566612876a51b060338916ab8e8cda3e8a21dd3b027e08e4be5a",
  "zones_v2.json": "3181ce3a3ae916f99d03887dc51526e585d42b0703908ddc008e7e0a7879610e"
}
```
Beklenen model ağırlığı: `7d7fb8d822aaf241f1be2f002f3bff9e72b66331e941408c4ef77d4e1d2900ce`

## 1. GPU ve bellek kontrolü

In [ ]:
import torch, psutil
if not torch.cuda.is_available():
    raise RuntimeError("GPU YOK. Calisma zamani -> Calisma zamani turunu degistir -> T4 GPU secip yeniden baslatin.")
print("GPU :", torch.cuda.get_device_name(0))
print(f"RAM : {psutil.virtual_memory().total / 1e9:.1f} GB (z=22 icin ~9 GB gerekir)")

## 2. Kütüphane
`segmentation-models-pytorch` yereldeki sürümle (**0.5.0**) aynı olmalı.

In [ ]:
!pip install -q segmentation-models-pytorch==0.5.0

## 3. Kod ve bölgeler (depodan birebir kopya)

In [ ]:
%%writefile inference.py
import os
from pathlib import Path

import torch
import cv2
import numpy as np
import segmentation_models_pytorch as smp
from huggingface_hub import hf_hub_download


DEFAULT_ROAD_MODEL_REPO = "Utbird/dispath_optimized_mitb4_focal_dice30"
DEFAULT_ROAD_MODEL_FILENAME = "optimized_mitb4_focal_dice30.pth"


def _ensure_model_file(model_path):
    resolved_path = Path(model_path)
    if resolved_path.exists():
        return resolved_path

    if resolved_path.name != DEFAULT_ROAD_MODEL_FILENAME:
        return resolved_path

    resolved_path.parent.mkdir(parents=True, exist_ok=True)
    downloaded_path = hf_hub_download(
        repo_id=DEFAULT_ROAD_MODEL_REPO,
        filename=DEFAULT_ROAD_MODEL_FILENAME,
        local_dir=str(resolved_path.parent),
    )
    return Path(downloaded_path)


_model_cache = {}

def load_simple_model(model_path):
    cache_key = str(model_path)
    if cache_key in _model_cache:
        return _model_cache[cache_key]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = smp.Segformer(encoder_name="mit_b4", encoder_weights=None, in_channels=3, classes=1).to(device)
    resolved_model_path = _ensure_model_file(model_path)
    if resolved_model_path.exists():
        model.load_state_dict(torch.load(resolved_model_path, map_location=device))
        model.eval()
        _model_cache[cache_key] = (model, device)
        return model, device
    return None, device


def _create_weight_window(size):
    """Merkeze doğru artan ağırlık penceresi oluşturur (overlap blending için)."""
    x = np.linspace(0, 1, size)
    # Kenarlardan merkeze doğru yumuşak geçiş
    weight_1d = np.minimum(x, 1 - x)
    weight_1d = np.clip(weight_1d * 4, 0, 1)  # Daha keskin geçiş
    weight_2d = np.outer(weight_1d, weight_1d).astype(np.float32)
    return weight_2d


def _postprocess_mask(pred_mask_binary, level=1):
    """
    Morfolojik post-processing.
    level=0: Kapalı (ham çıktı)
    level=1: Hafif (sadece çok küçük gürültü temizlenir)
    level=2: Güçlü (agresif temizlik)
    """
    if level == 0:
        return pred_mask_binary

    cleaned = pred_mask_binary.copy()

    if level == 1:
        # Hafif: sadece çok küçük noktaları temizle
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
        min_area = 30
    else:  # level >= 2
        # Güçlü: agresif temizlik (yeni model için ideal)
        kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel_open)
        kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_close)
        min_area = 150

    # Küçük bileşenleri sil
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(cleaned, connectivity=8)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] < min_area:
            cleaned[labels == i] = 0

    return cleaned


def run_inference(img, road_mask_binary, model, device, damage_booster, threshold,
                  use_imagenet_norm=False, postprocess_level=1, progress_callback=None):
    """
    Runs the Segformer model with overlapping patches and blending.

    Args:
        use_imagenet_norm: True ise ImageNet normalize uygular (yeni model için).
        postprocess_level: 0=Kapalı, 1=Hafif, 2=Güçlü
        progress_callback: Opsiyonel `fn(done_patches, total_patches)` -- patch
            dongusu CPU'da dakikalarca surebildigi icin cagiran tarafin gercek
            ilerleme gosterebilmesini saglar. Hicbir sey donmez, hatalari
            yutulur (ilerleme raporu analizi bozmamali).
    """
    w, h = img.size
    img_np = np.array(img.convert("RGB"))

    patch_size = 512
    stride = 256  # %50 overlap — patch sınır artefaktlarını ortadan kaldırır

    # ImageNet normalizasyon değerleri
    imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
    imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)

    # Ağırlık penceresi (kenarlar düşük, merkez yüksek)
    weight_window = _create_weight_window(patch_size)

    # Padding
    pad_h = max(patch_size - h, (stride - (h - patch_size) % stride) % stride)
    pad_w = max(patch_size - w, (stride - (w - patch_size) % stride) % stride)
    img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')

    padded_h, padded_w = img_padded.shape[:2]
    raw_probs = np.zeros((padded_h, padded_w), dtype=np.float32)
    weight_map = np.zeros((padded_h, padded_w), dtype=np.float32)

    y_positions = list(range(0, padded_h - patch_size + 1, stride))
    x_positions = list(range(0, padded_w - patch_size + 1, stride))
    total_patches = max(1, len(y_positions) * len(x_positions))
    done_patches = 0

    for y in y_positions:
        for x in x_positions:
            chunk = img_padded[y:y + patch_size, x:x + patch_size, :]

            input_tensor = torch.from_numpy(chunk).permute(2, 0, 1).float() / 255.0
            if use_imagenet_norm:
                input_tensor = (input_tensor - imagenet_mean.cpu()) / imagenet_std.cpu()
            input_tensor = input_tensor.unsqueeze(0).to(device)

            with torch.no_grad():
                output = model(input_tensor)
                chunk_pred = torch.sigmoid(output).squeeze().cpu().numpy()

            # Ağırlıklı toplama (blending)
            raw_probs[y:y + patch_size, x:x + patch_size] += chunk_pred * weight_window
            weight_map[y:y + patch_size, x:x + patch_size] += weight_window

            done_patches += 1
            if progress_callback is not None:
                try:
                    progress_callback(done_patches, total_patches)
                except Exception:
                    pass

    # Normalize (ağırlıklara böl)
    raw_probs = raw_probs / np.maximum(weight_map, 1e-6)

    # Padding'i kaldır
    raw_probs = raw_probs[:h, :w]

    # Boosted probabilities & threshold
    boosted_probs = np.clip(raw_probs * damage_booster, 0, 1)
    pred_mask_binary = (boosted_probs > threshold).astype(np.uint8)

    # Post-processing: morfolojik temizlik
    pred_mask_binary = _postprocess_mask(pred_mask_binary, level=postprocess_level)

    intersection = cv2.bitwise_and(pred_mask_binary, road_mask_binary)
    return raw_probs, boosted_probs, pred_mask_binary, intersection, img_np

In [ ]:
%%writefile qm_geo.py
"""assembly.bbox_from_center'in birebir kopyasi (defter uretici tarafindan gomuldu)."""
import math


def bbox_from_center(lat, lon, radius_km):
    lat_delta = radius_km / 111.0
    lon_delta = radius_km / (111.0 * max(math.cos(math.radians(lat)), 0.2))
    return (
        lon - lon_delta,
        lat - lat_delta,
        lon + lon_delta,
        lat + lat_delta,
    )

In [ ]:
%%writefile make_model_masks.py
"""Senaryo bolgeleri icin SegFormer hasar maskesi uretir (Colab T4 icin).

NEDEN AYRI BIR ADIM
    Model tabanli kapanma, modelin deprem sonrasi goruntuyu dogal
    cozunurlugune yakin gormesini gerektiriyor (J-20: z=15 / 3,86 m/px'te
    model hasari neredeyse hic gormuyor, z=18 / 0,48 m/px'te 25 kat fazla
    goruyor). 3 km yaricapli bir bolge z=18'de ~12 500 x 12 500 piksel ve
    ~2 300 yama demek; kullanicinin bilgisayarinda GPU olmadigi icin bu
    bolge basina saatler suruyor. Cikarim burada, GPU'da, BIR KEZ yapilir;
    cikan maskeler benchmark/data/model_masks/<BOLGE>/ altina konur ve
    senaryo (`full_scenario.py --closure-source model`) onlarla yerelde
    hizla kosar.

AYNI KOD, AYNI PARAMETRELER
    Cikarim, backend'in kendi `run_inference` fonksiyonuyla yapilir
    (apps/road_damage/utils/inference.py -- Colab defterine birebir
    kopyalanir) ve /analyze'in varsayilan parametreleri kullanilir.
    Model agirligi SHA-256 ile dogrulanir; farkliysa kosu durur.

CIKTI (bolge basina)
    damage_mask.png   0/255  -- modelin hasar tahmini (goruntu olmayan
                               piksellerde sifirlanir)
    valid_mask.png    0/255  -- deprem sonrasi goruntunun GERCEKTEN
                               bulundugu pikseller
    preview.jpg              -- goz kontrolu icin kucultulmus bindirme
    meta.json                -- sinirlar, zoom, kullanilan sahneler, model
                               parmak izi, parametreler, sureler

Yerel deneme (kucuk alan, CPU):
    python benchmark/colab/make_model_masks.py --zones H-Z1 --radius-km 0.15 --out <klasor>
"""
import argparse
import concurrent.futures as cf
import hashlib
import io
import json
import math
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import requests
from PIL import Image

HERE = Path(__file__).resolve().parent
REPO_BACKEND = HERE.parent.parent      # benchmark/colab -> QuakeMindBackend
IN_REPO = (REPO_BACKEND / "apps" / "road_damage").is_dir()
if IN_REPO:  # yerel depo
    sys.path.insert(0, str(REPO_BACKEND))
    from apps.road_damage.utils.inference import (  # noqa: E402
        DEFAULT_ROAD_MODEL_FILENAME, DEFAULT_ROAD_MODEL_REPO, load_simple_model, run_inference,
    )
    from apps.road_damage.utils.assembly import bbox_from_center  # noqa: E402
    LOCAL_MODEL_PATH = REPO_BACKEND / "apps" / "road_damage" / "models" / DEFAULT_ROAD_MODEL_FILENAME
else:  # Colab: defterin yanina yazilan birebir kopyalar
    from inference import (  # noqa: E402
        DEFAULT_ROAD_MODEL_FILENAME, DEFAULT_ROAD_MODEL_REPO, load_simple_model, run_inference,
    )
    from qm_geo import bbox_from_center  # noqa: E402
    LOCAL_MODEL_PATH = HERE / "models" / DEFAULT_ROAD_MODEL_FILENAME

EVENT_DATE = "2023-02-06"          # Kahramanmaras depremleri
POST_EVENT_END = "2023-06-30"      # "deprem sonrasi" penceresinin sonu
ZOOM = 18                          # ~0,48 m/px (36-38 derece enlemde)
TILE = 256
OAM_META_URL = "https://api.openaerialmap.org/meta"
USER_AGENT = "QuakeMind-research/1.0 (TUBITAK 2209-A; model mask generation)"

# /analyze'in varsayilanlari (fastapi_app.RoadDamageRequest / worker.py) --
# canli sistemle AYNI karar esigi kullanilsin.
INFERENCE_PARAMS = {
    "damage_booster": 3.5,
    "threshold": 0.40,
    "use_imagenet_norm": True,
    "postprocess_level": 2,
}


# --------------------------------------------------------------------------
# Karo matematigi (Web Mercator, XYZ)
# --------------------------------------------------------------------------
def deg2tile(lat, lon, z):
    n = 2 ** z
    x = (lon + 180.0) / 360.0 * n
    y = (1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n
    return x, y


def tile2deg(x, y, z):
    """(x, y) karosunun KUZEYBATI kosesi."""
    n = 2 ** z
    lon = x / n * 360.0 - 180.0
    lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / n))))
    return lat, lon


# --------------------------------------------------------------------------
# OpenAerialMap
# --------------------------------------------------------------------------
def search_post_event_scenes(bbox, session):
    """Bolgeyi kesen deprem sonrasi sahneler. Siralama deterministiktir:
    once Maxar (D-01: yayinda kullanilabilir acik veri), sonra en ERKEN
    tarih (moloz henuz kaldirilmamis), sonra kimlik."""
    west, south, east, north = bbox
    params = {
        "bbox": f"{west},{south},{east},{north}",
        "acquisition_from": EVENT_DATE,
        "acquisition_to": POST_EVENT_END,
        "limit": 100,
    }
    resp = session.get(OAM_META_URL, params=params, timeout=60)
    resp.raise_for_status()
    scenes = []
    for item in resp.json().get("results", []):
        tms = item.get("tms") or (item.get("properties") or {}).get("tms")
        date = item.get("acquisition_start") or item.get("acquisition_end") or ""
        if not tms or date[:10] < EVENT_DATE:
            continue
        scenes.append({
            "id": item.get("_id") or item.get("uuid"),
            "title": item.get("title"),
            "provider": item.get("provider"),
            "date": date[:10],
            "tms": tms,
        })
    scenes.sort(key=lambda s: (0 if "maxar" in (s["provider"] or "").lower() else 1,
                               s["date"], str(s["id"])))
    return scenes


def fetch_tile(session, tms, z, x, y, retries=4):
    """RGBA karo -> (rgb uint8, valid uint8) ya da None (sahne bu karoyu
    kapsamiyor). OAM, kapsama disini seffaf/bos karo olarak dondurur."""
    url = tms.replace("{z}", str(z)).replace("{x}", str(x)).replace("{y}", str(y))
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=60)
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
            continue
        if r.status_code == 404:
            return None
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2.0 * (attempt + 1))
            continue
        if r.status_code != 200 or not r.content:
            return None
        img = Image.open(io.BytesIO(r.content))
        rgba = np.array(img.convert("RGBA"))
        if rgba.shape[:2] != (TILE, TILE):
            rgba = np.array(img.convert("RGBA").resize((TILE, TILE), Image.BILINEAR))
        valid = (rgba[:, :, 3] > 0).astype(np.uint8)
        if not valid.any():
            return None
        return rgba[:, :, :3], valid
    return None


def build_mosaic(bbox, scenes, session, z=ZOOM, workers=16, progress=None):
    """Bolge sinirlarini kaplayan karolari, sahne onceligine gore piksel
    piksel doldurarak birlestirir. Bir pikseli ilk dolduran sahne kazanir."""
    west, south, east, north = bbox
    fx0, fy0 = deg2tile(north, west, z)
    fx1, fy1 = deg2tile(south, east, z)
    x0, y0, x1, y1 = int(fx0), int(fy0), int(fx1), int(fy1)
    nx_, ny_ = x1 - x0 + 1, y1 - y0 + 1
    H, W = ny_ * TILE, nx_ * TILE
    rgb = np.zeros((H, W, 3), dtype=np.uint8)
    valid = np.zeros((H, W), dtype=np.uint8)
    tiles_from_scene = {s["id"]: 0 for s in scenes}

    def work(tx, ty):
        tile_rgb = np.zeros((TILE, TILE, 3), dtype=np.uint8)
        tile_valid = np.zeros((TILE, TILE), dtype=np.uint8)
        used = []
        for sc in scenes:
            got = fetch_tile(session, sc["tms"], z, tx, ty)
            if got is None:
                continue
            s_rgb, s_valid = got
            fill = (s_valid > 0) & (tile_valid == 0)
            if fill.any():
                tile_rgb[fill] = s_rgb[fill]
                tile_valid[fill] = 1
                used.append(sc["id"])
            if tile_valid.all():
                break
        return tx, ty, tile_rgb, tile_valid, used

    jobs = [(tx, ty) for ty in range(y0, y1 + 1) for tx in range(x0, x1 + 1)]
    done = 0
    with cf.ThreadPoolExecutor(max_workers=workers) as pool:
        for tx, ty, t_rgb, t_valid, used in pool.map(lambda j: work(*j), jobs):
            oy, ox = (ty - y0) * TILE, (tx - x0) * TILE
            rgb[oy:oy + TILE, ox:ox + TILE] = t_rgb
            valid[oy:oy + TILE, ox:ox + TILE] = t_valid
            for sid in used:
                tiles_from_scene[sid] += 1
            done += 1
            if progress and (done % 200 == 0 or done == len(jobs)):
                progress(done, len(jobs))

    n_lat, w_lon = tile2deg(x0, y0, z)
    s_lat, e_lon = tile2deg(x1 + 1, y1 + 1, z)
    bounds = {"west": w_lon, "south": s_lat, "east": e_lon, "north": n_lat}
    return rgb, valid, bounds, tiles_from_scene, (nx_, ny_)


# --------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_verified_model(expected_sha256):
    model, device = load_simple_model(str(LOCAL_MODEL_PATH))
    if model is None:
        raise RuntimeError(f"Model yuklenemedi: {LOCAL_MODEL_PATH}")
    actual = sha256_of(LOCAL_MODEL_PATH)
    if expected_sha256 and actual != expected_sha256:
        raise RuntimeError(
            "MODEL AGIRLIGI FARKLI -- kosu durduruldu.\n"
            f"  beklenen: {expected_sha256}\n  bulunan : {actual}\n"
            "Maskeler yereldeki modelle ayni agirlikla uretilmeli; aksi halde "
            "makaledeki sonuclar baska bir modele ait olur.")
    return model, device, actual


def library_versions():
    import importlib.metadata as md
    out = {"python": platform.python_version()}
    for pkg in ("torch", "segmentation-models-pytorch", "timm", "numpy", "opencv-python",
                "opencv-python-headless", "Pillow"):
        try:
            out[pkg] = md.version(pkg)
        except Exception:
            pass
    return out


def preview_base(img, max_side=2048):
    """Tam cozunurluklu mozaigin kucuk kopyasi -- cikarimdan ONCE alinir ki
    buyuk RGB dizisi cikarim sirasinda bellekte tutulmasin (z=22'de tek bir
    bolge ~370 MP; Colab'in ~12 GB RAM'inde tasma riski)."""
    w, h = img.size
    scale = min(1.0, max_side / max(h, w))
    size = (max(1, int(w * scale)), max(1, int(h * scale)))
    return img.resize(size, Image.BILINEAR)


def save_preview(base, damage, valid, path):
    size = base.size
    arr = np.array(base).astype(np.float32)
    dmg = np.array(Image.fromarray(damage * 255).resize(size, Image.NEAREST)) > 0
    val = np.array(Image.fromarray(valid * 255).resize(size, Image.NEAREST)) > 0
    arr[~val] *= 0.25                          # goruntu olmayan alan koyu
    arr[dmg] = arr[dmg] * 0.3 + np.array([255, 0, 0]) * 0.7   # hasar kirmizi
    Image.fromarray(arr.clip(0, 255).astype(np.uint8)).save(path, quality=85)


# --------------------------------------------------------------------------
# Ana akis
# --------------------------------------------------------------------------
def process_zone(zone, out_root, model, device, model_sha, radius_km, session,
                 zoom=ZOOM, mask_suffix=""):
    zid = zone["id"]
    mask_id = f"{zid}{mask_suffix}"
    lat, lon = zone["center"]
    radius = radius_km if radius_km is not None else zone["radiusKm"]
    if zone.get("bbox") and radius_km is None:
        # Drone sahnesi bolgeleri: maske sahnenin KENDI kapsama kutusu uzerinde
        # uretilir (bolge merkezinden turetilen kare yerine).
        bbox = tuple(zone["bbox"])
    else:
        bbox = bbox_from_center(lat, lon, radius)
    print(f"\n=== {mask_id} {zone['label']}  (z={zoom}) ===", flush=True)

    scenes = search_post_event_scenes(bbox, session)
    if zone.get("sceneIds"):
        # Sahne SABITLENDI: yalniz belirtilen sahneler, verilen sirayla. Drone
        # bolgelerinde Maxar'in onceligi istenmez -- farkli tarih ve 10 kat
        # kaba cozunurluk ayni maskeye karismasin.
        by_id = {sc["id"]: sc for sc in scenes}
        scenes = [by_id[i] for i in zone["sceneIds"] if i in by_id]
    if not scenes:
        print("  ATLANDI: deprem sonrasi sahne yok.", flush=True)
        return {"zoneId": mask_id, "status": "skipped", "reason": "no post-event scenes"}
    for s in scenes:
        print(f"  sahne: {s['date']}  {s['provider']:<10} {s['title']}", flush=True)

    t0 = time.perf_counter()
    rgb, valid, bounds, per_scene, (ntx, nty) = build_mosaic(
        bbox, scenes, session, z=zoom,
        progress=lambda d, n: print(f"  karo {d}/{n}", flush=True))
    t_fetch = time.perf_counter() - t0
    h, w = valid.shape
    valid_frac = float(valid.mean())
    print(f"  mozaik {w}x{h} px ({ntx}x{nty} karo), goruntu kapsamasi %{100 * valid_frac:.1f}", flush=True)
    if valid_frac == 0.0:
        return {"zoneId": mask_id, "status": "skipped", "reason": "scenes returned no pixels"}

    image = Image.fromarray(rgb)
    del rgb  # PIL kendi kopyasini tutuyor; buyuk dizi cikarim sirasinda bellekte kalmasin
    small = preview_base(image)

    last = [0]

    def on_patch(done, total):
        pct = int(100 * done / max(1, total))
        if pct >= last[0] + 10 or done == total:
            last[0] = pct
            print(f"  cikarim %{pct} ({done}/{total} yama)", flush=True)

    t1 = time.perf_counter()
    _raw, _boosted, pred, _inter, _img = run_inference(
        image, np.zeros((h, w), dtype=np.uint8), model, device,
        progress_callback=on_patch, **INFERENCE_PARAMS)
    t_inf = time.perf_counter() - t1
    del _raw, _boosted, _inter, _img

    # Goruntu olmayan piksellerde tahmin anlamsizdir (siyah dolgu) -- sifirla.
    damage = (pred > 0).astype(np.uint8) & valid

    del image
    out = out_root / mask_id
    out.mkdir(parents=True, exist_ok=True)
    Image.fromarray(damage * 255).save(out / "damage_mask.png", optimize=True)
    Image.fromarray(valid * 255).save(out / "valid_mask.png", optimize=True)
    save_preview(small, damage, valid, out / "preview.jpg")

    mid_lat = (bounds["south"] + bounds["north"]) / 2.0
    mpp = 40075016.686 * math.cos(math.radians(mid_lat)) / (TILE * 2 ** zoom)
    meta = {
        "maskId": mask_id,
        "zoneId": zid, "zoneLabel": zone["label"], "city": zone["city"],
        "center": {"lat": lat, "lon": lon}, "radiusKm": radius,
        "zoneBbox": {"west": bbox[0], "south": bbox[1], "east": bbox[2], "north": bbox[3]},
        "bounds": bounds, "width": w, "height": h,
        "zoom": zoom, "metresPerPixel": round(mpp, 4),
        "sceneNativeGsdM": zone.get("sceneGsdM"),
        "projectionNote": ("Web Mercator karolari; yerel boru hatti enlemi dogrusal "
                           "izdusumler (canli /analyze ile ayni). 6 km'lik bolgede en kotu "
                           "sapma ~0,5 m (~1 piksel)."),
        "eventDate": EVENT_DATE, "postEventWindowEnd": POST_EVENT_END,
        "scenes": [{**{k: s[k] for k in ("id", "title", "provider", "date")},
                    "tilesUsed": per_scene.get(s["id"], 0)} for s in scenes],
        "validPixelFraction": round(valid_frac, 6),
        "damagePixelFraction": round(float(damage.mean()), 6),
        "damagePixelFractionOfObserved": round(float(damage.sum() / max(1, valid.sum())), 6),
        "model": {"repo": DEFAULT_ROAD_MODEL_REPO, "file": DEFAULT_ROAD_MODEL_FILENAME,
                  "sha256": model_sha, "architecture": "smp.Segformer(mit_b4, classes=1)"},
        "damageBooster": INFERENCE_PARAMS["damage_booster"],
        "threshold": INFERENCE_PARAMS["threshold"],
        "useImagenetNorm": INFERENCE_PARAMS["use_imagenet_norm"],
        "postProcessLevel": INFERENCE_PARAMS["postprocess_level"],
        "invalidPixelsZeroed": True,
        "inferenceDevice": str(device),
        "libraries": library_versions(),
        "timingsSec": {"fetch": round(t_fetch, 1), "inference": round(t_inf, 1)},
        "createdAt": datetime.now(timezone.utc).isoformat(),
    }
    (out / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  hasar: gorulen alanin %{100 * meta['damagePixelFractionOfObserved']:.2f}'i  |  "
          f"indirme {t_fetch:.0f}s, cikarim {t_inf:.0f}s  ->  {out}", flush=True)
    return {"zoneId": mask_id, "status": "done", "dir": str(out),
            "validPixelFraction": meta["validPixelFraction"],
            "damagePixelFractionOfObserved": meta["damagePixelFractionOfObserved"],
            "timingsSec": meta["timingsSec"]}


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--zones-json", default=None,
                    help="Bolge listesi (Colab). Verilmezse yerel scenario_dataset.py okunur.")
    ap.add_argument("--zones", nargs="*", default=None, help="Yalniz bu bolgeler (orn. H-Z1 K-Z1)")
    ap.add_argument("--radius-km", type=float, default=None,
                    help="Deneme icin bolge yaricapini ez (varsayilan: bolgenin kendi yaricapi)")
    ap.add_argument("--dataset", choices=["v2", "drone"], default="v2",
                    help="Yerel bolge listesi: v2 = Senaryo v2 (8 ilce), drone = Help.NGO drone sahneleri")
    ap.add_argument("--zoom", type=int, default=ZOOM,
                    help="Karo zoom'u (18 = ~0,48 m/px; drone sahnelerinin dogal siniri 22 = ~3 cm/px)")
    ap.add_argument("--mask-suffix", default="",
                    help="Cikti klasoru soneki; ayni bolgenin farkli zoom'larini yan yana tutmak icin (orn. -z20)")
    ap.add_argument("--out", default="model_masks")
    ap.add_argument("--expected-model-sha256", default=None)
    args = ap.parse_args(argv)

    if args.zones_json:
        zones = json.loads(Path(args.zones_json).read_text(encoding="utf-8"))
    elif args.dataset == "drone":
        from benchmark.scenario_dataset_drone import ZONES
        zones = ZONES
    else:
        from benchmark.scenario_dataset import ZONES
        zones = ZONES
    if args.zones:
        zones = [z for z in zones if z["id"] in set(args.zones)]
    if not zones:
        raise RuntimeError("Islenecek bolge yok.")

    model, device, model_sha = load_verified_model(args.expected_model_sha256)
    print(f"model hazir: {device}, sha256 {model_sha[:16]}...", flush=True)

    session = requests.Session()
    session.headers["User-Agent"] = USER_AGENT
    adapter = requests.adapters.HTTPAdapter(pool_connections=32, pool_maxsize=32)
    session.mount("https://", adapter)

    out_root = Path(args.out)
    out_root.mkdir(parents=True, exist_ok=True)
    manifest = []
    for zone in zones:
        try:
            manifest.append(process_zone(zone, out_root, model, device, model_sha, args.radius_km,
                                         session, args.zoom, args.mask_suffix))
        except Exception as e:  # bir bolge patlarsa digerleri yine uretilsin
            print(f"  HATA: {type(e).__name__}: {e}", flush=True)
            manifest.append({"zoneId": f"{zone['id']}{args.mask_suffix}", "status": "error",
                             "error": f"{type(e).__name__}: {e}"})
    (out_root / f"manifest{args.mask_suffix}.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

    print("\n=== OZET ===")
    for m in manifest:
        extra = ""
        if m["status"] == "done":
            extra = (f"kapsama %{100 * m['validPixelFraction']:.1f}, "
                     f"hasar %{100 * m['damagePixelFractionOfObserved']:.2f}")
        print(f"  {m['zoneId']:10} {m['status']:8} {extra or m.get('reason') or m.get('error', '')}")
    return 0 if all(m["status"] != "error" for m in manifest) else 1


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile zones_drone.json
[
  {
    "id": "H-D1",
    "city": "hatay",
    "label": "Defne - Elektrik Mah.",
    "radiusKm": 0.33,
    "bbox": [
      36.146753,
      36.195466,
      36.15189,
      36.199922
    ],
    "sceneIds": [
      "63eb78a0ca43600005f4d920"
    ],
    "sceneTitle": "Hatay - Defne - Elektrik Mah.",
    "sceneGsdM": 0.0341,
    "nativeZoom": 22,
    "center": [
      36.197694,
      36.149321
    ]
  },
  {
    "id": "H-D2",
    "city": "hatay",
    "label": "Defne - Turunclu / Akdeniz / Armutlu Mah.",
    "radiusKm": 0.39,
    "bbox": [
      36.142957,
      36.192684,
      36.15092,
      36.19543
    ],
    "sceneIds": [
      "63eb829fca43600005f4d927"
    ],
    "sceneTitle": "Hatay - Turinclu - Akdeniz Mah - Armutlu Mah.",
    "sceneGsdM": 0.0386,
    "nativeZoom": 22,
    "center": [
      36.194057,
      36.146939
    ]
  },
  {
    "id": "K-D1",
    "city": "kahramanmaras",
    "label": "Merkez - 12 Subat Stadyumu cevresi",
    "radiusKm": 0.4,
    "bbox": [
      36.917486,
      37.573071,
      36.923946,
      37.578173
    ],
    "sceneIds": [
      "63eb7f38ca43600005f4d924"
    ],
    "sceneTitle": "12 Subat Stadyumu, Merkez/Kahramanmaras",
    "sceneGsdM": 0.0376,
    "nativeZoom": 22,
    "center": [
      37.575622,
      36.920716
    ]
  }
]

In [ ]:
%%writefile zones_v2.json
[
  {
    "id": "H-Z1",
    "city": "hatay",
    "label": "Antakya - Defne",
    "radiusKm": 3.0,
    "center": [
      36.194057,
      36.146939
    ]
  },
  {
    "id": "H-Z2",
    "city": "hatay",
    "label": "Iskenderun",
    "radiusKm": 3.0,
    "center": [
      36.5875,
      36.17
    ]
  },
  {
    "id": "H-Z3",
    "city": "hatay",
    "label": "Kirikhan",
    "radiusKm": 3.0,
    "center": [
      36.4986,
      36.3606
    ]
  },
  {
    "id": "H-Z4",
    "city": "hatay",
    "label": "Samandag",
    "radiusKm": 3.0,
    "center": [
      36.085,
      35.98
    ]
  },
  {
    "id": "H-Z5",
    "city": "hatay",
    "label": "Belen",
    "radiusKm": 3.0,
    "center": [
      36.49,
      36.2
    ]
  },
  {
    "id": "K-Z1",
    "city": "kahramanmaras",
    "label": "Merkez (Onikisubat - Dulkadiroglu)",
    "radiusKm": 3.0,
    "center": [
      37.5756,
      36.9207
    ]
  },
  {
    "id": "K-Z2",
    "city": "kahramanmaras",
    "label": "Elbistan",
    "radiusKm": 3.0,
    "center": [
      38.205,
      37.195
    ]
  },
  {
    "id": "K-Z3",
    "city": "kahramanmaras",
    "label": "Pazarcik",
    "radiusKm": 3.0,
    "center": [
      37.489,
      37.3
    ]
  }
]

In [ ]:
import hashlib
expected = {"inference.py": "9204c92a668f52daec9f6178efe412c0661b80eb51b8a3fa4604179c8a6c2724", "make_model_masks.py": "3ebf39cf4536d81dd3b910fce612e7e17bd2ee120a1bda3f4527df40f5fd7b1e", "qm_geo.py": "199cf916abb00aa17d3238e5feca29f7d367f7ec957dce3546960d0f90fbba6a", "zones_drone.json": "6aa623217d28566612876a51b060338916ab8e8cda3e8a21dd3b027e08e4be5a", "zones_v2.json": "3181ce3a3ae916f99d03887dc51526e585d42b0703908ddc008e7e0a7879610e"}
for name, h in expected.items():
    got = hashlib.sha256((open(name, encoding="utf-8").read().rstrip("\n") + "\n").encode("utf-8")).hexdigest()
    assert got == h, f"{name} beklenenden farkli -- defteri depodan yeniden uretin"
print("Gomulu dosyalar dogrulandi.")

## 4. Drone sahneleri — z=21 (~6 cm/px)
Model ağırlığı HuggingFace'ten otomatik iner ve parmak izi kontrol edilir; farklıysa çalışma durur.

In [ ]:
import make_model_masks as mm
rc = mm.main(["--zones-json", "zones_drone.json", "--zoom", "21", "--mask-suffix=-z21",
              "--out", "model_masks", "--expected-model-sha256", "7d7fb8d822aaf241f1be2f002f3bff9e72b66331e941408c4ef77d4e1d2900ce"])
if rc != 0:
    raise RuntimeError("z=21: en az bir bolgede HATA var -- yukaridaki ciktiya bakin.")

### z=21 sonuçlarını hemen indir (z=22 çökerse elde kalsın)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("model_masks", "zip", ".", "model_masks")
files.download("model_masks.zip")

## 5. Drone sahneleri — z=22 (~3 cm/px, doğal çözünürlük)

In [ ]:
import make_model_masks as mm
rc = mm.main(["--zones-json", "zones_drone.json", "--zoom", "22", "--mask-suffix=-z22",
              "--out", "model_masks", "--expected-model-sha256", "7d7fb8d822aaf241f1be2f002f3bff9e72b66331e941408c4ef77d4e1d2900ce"])
if rc != 0:
    raise RuntimeError("z=22: en az bir bolgede HATA var -- yukaridaki ciktiya bakin.")

## 6. Göz kontrolü
Kırmızı: modelin hasar dediği yer. Koyu: görüntü olmayan alan.

In [ ]:
import json, glob
from IPython.display import Image as IPImage, display
for meta_path in sorted(glob.glob("model_masks/*/meta.json")):
    m = json.load(open(meta_path, encoding="utf-8"))
    print(f"{m['maskId']:10} z={m['zoom']}  {100*m['metresPerPixel']:.1f} cm/px  "
          f"kapsama %{100*m['validPixelFraction']:.1f}  hasar %{100*m['damagePixelFractionOfObserved']:.2f}  "
          f"sure indirme {m['timingsSec']['fetch']}s / cikarim {m['timingsSec']['inference']}s")
    display(IPImage(meta_path.replace("meta.json", "preview.jpg"), width=480))

## 7. Tümünü indir (z=21 + z=22)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("model_masks", "zip", ".", "model_masks")
files.download("model_masks.zip")

---
## (İsteğe bağlı) Senaryo v2 — 8 ilçe, z=18
Drone sahnelerinden bağımsız. Çalıştırmak için alttaki hücrenin başındaki `#` işaretlerini kaldır.

In [ ]:
# import make_model_masks as mm
# rc = mm.main(["--zones-json", "zones_v2.json", "--zoom", "18", "--out", "model_masks",
#               "--expected-model-sha256", "7d7fb8d822aaf241f1be2f002f3bff9e72b66331e941408c4ef77d4e1d2900ce"])